# ML Core Concepts — the Round-One Warm-Ups

Every ML depth round opens with the same five minutes: train/test/validation, overfitting, cross-validation, and the ML project lifecycle. These look easy and are exactly where mid-level candidates telegraph sloppiness ("I just did `train_test_split`"). The answers below are calibrated to senior-bar precision — and the two claims people fumble most (why we need a test set, and what CV actually estimates) are demonstrated in code.

## Q1 — Train / Validation / Test: What Each Set Is *For*

The senior-version answer, in one breath each:

- **Training set** — what the model *learns from* (fit weights).
- **Validation set** — what *you* learn from: model selection, hyperparameters, early stopping. Every decision informed by it leaks a little of it.
- **Test set** — a single-use simulation of unseen data. It answers exactly one question, once: *how will this exact pipeline perform in production?* Used twice, it's a validation set wearing a costume.

**The rule that gets quoted back at you:** you may look at the test set's *distribution* during EDA in many teams' practice, but never its joint relationship to the target, and the final metric on it is consumed once. If you need repeated comparisons, that's validation's job — hence CV.

## Q2 — Overfitting: Definition, Diagnosis, and the Three Cures

**Definition to say:** the model captures noise/peculiarities of the training sample that don't generalize — formally, training error keeps falling while generalization error rises. **Diagnosis:** the gap between train and validation performance, not the absolute validation number.

In [1]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

rng = np.random.default_rng(0)
# signal + pure noise dimensions: deeper trees can memorize the noise
X, y = make_classification(n_samples=600, n_features=20, n_informative=3,
                           n_redundant=2, random_state=0)

Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.3, random_state=1)

print(f"{'depth':>6} {'train_acc':>10} {'val_acc':>9} {'gap':>7}")
for depth in (1, 3, 5, 8, 12, None):
    t = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(Xtr, ytr)
    tr = t.score(Xtr, ytr); va = t.score(Xva, yva)
    print(f"{str(depth):>6} {tr:>10.3f} {va:>9.3f} {tr-va:>7.3f}")
print()
print("read it: gap widens with depth; validation peaks then decays — the overfitting signature")

 depth  train_acc   val_acc     gap
     1      0.714     0.711   0.003
     3      0.788     0.778   0.010
     5      0.900     0.833   0.067
     8      0.967     0.800   0.167
    12      1.000     0.861   0.139
  None      1.000     0.861   0.139

read it: gap widens with depth; validation peaks then decays — the overfitting signature


**The three cures, grouped the way interviewers like:**
1. **More data / less capacity** — regularize (L1/L2, max_depth, min_samples_leaf, dropout), or get more training samples.
2. **Better estimation of what generalizes** — cross-validation for selection, early stopping on validation loss.
3. **Ensembling** — averaging decorrelated errors (bagging) or sequentially correcting (boosting) — see the Trees notebook.

The measured version of this experiment, on more model families, lives in the repo's [Bias-Variance Tradeoff notebook](../../2.%20Gaining%20Conceptual%20depth/8.%20Bias-Variance%20Tradeoff/Bias-Variance%20Tradeoff.ipynb).

## Q3 — Cross-Validation: What It Actually Estimates

**The precise answer interviewers wait for:** k-fold CV estimates **the expected performance of the *model-building procedure*** — "fit this pipeline on a sample of this size" — not the performance of any single fitted model. Each fold's model is discarded (in standard use); the ± std across folds is your uncertainty about the *procedure*.

**Mechanics to rattle off:** k=5 or 10 standard; each point used for validation exactly once and training k−1 times; stratified for classification (class proportions preserved); **for the CV estimate to be honest, everything data-dependent — scaling, imputation, feature selection — must be inside the fold**, i.e., inside a `Pipeline`.

In [2]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")
print("fold scores :", scores.round(3))
print(f"mean +/- std: {scores.mean():.3f} +/- {scores.std():.3f}")
print("report it WITH the std — a single fold number is noise")

fold scores : [0.675 0.608 0.642 0.558 0.717]
mean +/- std: 0.640 +/- 0.054
report it WITH the std — a single fold number is noise


**The variants to name-drop with one use case each:** `GroupKFold` (rows share an entity — patients, users), `TimeSeriesSplit` (no shuffling; train on past only), `StratifiedKFold` (imbalance), nested CV (unbiased tuning *and* estimation — expensive but correct). Say: *"the split scheme must respect how the data was generated."*

## Q4 — Data Distributions Shift: The Three Kinds of Drift

This question separates people who've operated models from people who've only fit them:

- **Covariate shift** — P(X) changes, P(y|X) doesn't. Example: a new marketing channel brings younger users. The model's mapping still holds, but it's being applied outside its dense support — performance degrades at the edges. Fix: retrain on recent data, importance-weight.
- **Label shift** — P(y) changes, P(y|X) doesn't. Example: fraud rate doubles after a viral attack. The classifier's ranking stays valid; the *threshold/calibration* is stale. Fix: recalibrate thresholds, adjust priors.
- **Concept drift** — P(y|X) itself changes. Example: the definition of "churn" changes when a new competitor makes customers leave earlier. The mapping is wrong; nothing but retraining and monitoring the *feature→target relation* helps.

**Monitoring answer:** track input distributions (PSI, KL divergence per feature) *and* performance where labels arrive; alert on distribution change even before labels exist.

In [3]:
# population stability index — the drift metric interviewers name-check
def psi(expected, actual, bins=10):
    edges = np.quantile(expected, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, bins=edges)[0] / len(expected)
    a = np.histogram(actual, bins=edges)[0] / len(actual)
    e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)   # avoid log(0)
    return float(np.sum((a - e) * np.log(a / e)))

rng = np.random.default_rng(1)
train_scores = rng.normal(0.5, 0.15, 5000)
same = rng.normal(0.5, 0.15, 2000)
shifted = rng.normal(0.62, 0.15, 2000)          # covariate shift on a feature

print(f"PSI (no drift)  : {psi(train_scores, same):.4f}   <- <0.1 stable")
print(f"PSI (shifted)   : {psi(train_scores, shifted):.4f}   <- >0.25 major shift, alert")
print("rule of thumb: <0.1 stable | 0.1-0.25 watch | >0.25 act")

PSI (no drift)  : 0.0068   <- <0.1 stable
PSI (shifted)   : 0.6146   <- >0.25 major shift, alert
rule of thumb: <0.1 stable | 0.1-0.25 watch | >0.25 act


## Q5 — The ML Project Lifecycle (Asked as 'Walk Me Through Your Process')

Answer as a pipeline with feedback loops, not a checklist — the loops are the senior signal:

1. **Frame:** business objective → measurable proxy. ("Reduce churn" → predicted P(cancel in 30d) with a retention-action cost model.)
2. **Data:** availability, representativeness, label quality, volume, freshness. *The framing dies here if labels don't exist — that's a finding, not a failure.*
3. **EDA + feature engineering:** decisions driven by data, leakage guards in mind.
4. **Baseline:** dumb model + dumb features. Everything after is judged relative to it.
5. **Model development:** 2–3 candidate families, CV, tuning — inside pipelines.
6. **Evaluation:** offline metrics aligned to cost (see the Metrics notebook), error analysis by segment, calibration if downstream decisions matter.
7. **Deployment:** serving, latency, monitoring for drift and performance. ([Model Serving](../../3.%20Learn%20Practical%20Concepts/17.%20Model%20Serving/Model%20Serving.ipynb) covers the patterns.)
8. **Monitor + iterate:** drift alarms, scheduled retraining, feedback from users; loop back to 1 when the business question moves.

**The sentence that lands:** *"Most of the risk in an ML project is concentrated in steps 1–2 — wrong framing and bad labels can't be fixed by modeling, and no modeling success downstream compensates for them."*

## Q6 — Supervised vs Unsupervised vs Semi/Self-Supervised, in 2026 Terms

- **Supervised** — labeled pairs; the workhorse of business ML (churn, fraud, forecasting).
- **Unsupervised** — structure without labels: clustering, dimensionality reduction, density estimation, anomaly detection. Say a use case: *segmentation → different retention strategies per segment.*
- **Semi-supervised** — few labels, lots of unlabeled; label propagation, self-training.
- **Self-supervised** — labels manufactured from the data itself: next-token prediction, masked modeling, contrastive learning. The engine of the foundation-model era — GPT, BERT, CLIP, and today's LLMs are all self-supervised pretraining + (optional) fine-tuning.
- **RL(D)** — learn from delayed reward via interaction; production ML sees it in recommendation (bandits) more than robotics.

Interviewers increasingly probe the last one: *"why did self-supervision take over?"* — because unlabeled data is effectively unbounded and internet-scale, while labels cap at human effort; representation learning transfers, labels don't.

## Q7 — Parametric vs Non-Parametric (and Which You'd Pick)

**Parametric** models fix a functional form with a finite parameter count (linear/logistic regression, neural nets) — fast, sample-efficient, strong bias, brittle if the form is wrong. **Non-parametric** let complexity grow with data (KNN, trees, kernel SVM, Gaussian processes) — flexible, but memory- and compute-hungry and more variance-prone. Deep nets are *parametric but flexible* — a good nuance to volunteer. The pick: start parametric for interpretability and baseline; go non-parametric/tree ensembles when nonlinearity dominates and data is tabular and plentiful.

## Q8 — Quick-Fire: The Definitions Round

| Question | The 15-second answer |
| --- | --- |
| Batch vs online learning? | Retrain on accumulated data periodically vs update incrementally per example (streaming); online fits drift and scale, batch fits auditability |
| Instance vs feature? | Row vs column; an instance is one observation, a feature one measured attribute |
| Generative vs discriminative? | Model P(X,y) then derive P(y|X) vs model P(y|X) directly; generative can synthesize data and use unlabeled X, discriminative is usually more accurate for classification |
| Loss vs metric? | Loss is what training optimizes (differentiable); metric is what the business cares about (may be non-differentiable, thresholded, constrained) |
| What's a heuristic baseline? | A rule-based model ("if amount > 10x average: flag") — always build it; sometimes it wins, and it tells you how much the ML is really buying |
| Feature store? | A curated, versioned repository of reusable feature computations for training and serving — kills training/serving skew; name Feast as the OSS example |
| Experiment tracking? | MLflow / W&B — params, metrics, artifacts per run; "if it isn't logged, it didn't happen" |
| What's model versioning? | Version code + data + hyperparams + artifacts together so any result is reproducible; a model without its training data version is unauditable |